# Этап 19 — Протокол проверки нового признака

## Исследовательский вопрос

**Как должен быть устроен единый воспроизводимый процесс проверки нового признака, чтобы сравнение с текущей моделью оставалось честным, не возникала временная утечка, а решение о принятии признака учитывало не только общее качество, но и характер ошибок?**

## Почему этот вопрос проверяется сейчас

Предыдущие этапы показали, что дальнейший поиск моделей на неизменных 47 признаках имеет низкий ожидаемый информационный эффект, а историческое добавление современных внешних данных в текущий датасет заблокировано из-за отсутствия надёжной временной привязки строк.

Поэтому следующий практический шаг — зафиксировать единый протокол, по которому любой будущий новый признак будет проходить одинаковую проверку до включения в рабочую модель.

## Что остаётся неизменным

- целевая переменная: `DefMark`;
- идентификатор: `INN`;
- текущий набор: 47 разрешённых признаков;
- `Q_B1_norm` и `Q_B2_norm` не используются как predictors;
- разбиение working / final test не меняется;
- final test не используется для выбора признака;
- базовая модель и сохранённые OOF-результаты остаются точкой сравнения;
- один эксперимент проверяет только одно контролируемое изменение.

## Предлагаемая последовательность проверки

1. Источник и происхождение признака.
2. Проверка допустимости по времени.
3. Фиксация единственного изменения относительно baseline.
4. Проверка на той же working-выборке и по тому же протоколу.
5. Сравнение Gini / ROC-AUC, PR-AUC, Recall, Precision и F1.
6. Анализ ложноположительных и ложноотрицательных ошибок.
7. Проверка влияния на зону дополнительной проверки.
8. Отдельная проверка поведения 805 сложных дефолтов.
9. Решение: принять признак, отклонить или признать evidence недостаточным.

На этом этапе новый внешний признак ещё не добавляется и модель не переобучается. Сначала фиксируется сам проверяемый протокол.

## 1. Допуск нового признака к эксперименту

Прежде чем новый признак попадёт в обучение модели, необходимо проверить не его качество, а **право участвовать в эксперименте**.

Сначала отвечаем на три вопроса:

1. **Откуда получен признак?**
2. **Что именно он означает и как формируется?**
3. **Был ли он реально доступен на момент принятия решения по соответствующему клиенту?**

Это важно именно сейчас, потому что этап 17 уже показал ограничение текущего исторического датасета: надёжной построчной даты наблюдения или решения нет. Поэтому современные данные из внешнего источника нельзя просто присоединить к старым строкам по `INN` и считать историческим признаком.

### Правило допуска

Новый признак допускается к проверке качества только тогда, когда одновременно подтверждены:

- источник данных;
- правило формирования признака;
- единица наблюдения;
- временная привязка;
- отсутствие использования будущей информации;
- возможность воспроизвести значение признака для каждой строки исследуемой выборки.

Если хотя бы одно из этих условий не подтверждено, эксперимент с качеством модели **не запускается**.

### Что остаётся неизменным

- `DefMark` остаётся целевой переменной;
- `INN` остаётся идентификатором;
- текущие 47 разрешённых признаков не изменяются;
- `Q_B1_norm` и `Q_B2_norm` не возвращаются в predictors;
- working / final test не пересоздаются;
- final test не используется;
- модель не обучается на этом шаге.

In [2]:
# 1.1. Формальный допуск нового признака к эксперименту

from dataclasses import dataclass, asdict
from typing import Optional


@dataclass
class КарточкаНовогоПризнака:
    название: str
    источник: Optional[str]
    описание: Optional[str]
    правило_формирования: Optional[str]
    единица_наблюдения: Optional[str]
    временная_привязка: Optional[str]
    доступен_на_момент_решения: Optional[bool]
    использует_будущую_информацию: Optional[bool]
    воспроизводим_для_строк: Optional[bool]


def проверить_допуск(карточка: КарточкаНовогоПризнака) -> dict:
    проверки = {
        "Источник подтверждён": bool(карточка.источник),
        "Описание подтверждено": bool(карточка.описание),
        "Правило формирования подтверждено": bool(карточка.правило_формирования),
        "Единица наблюдения подтверждена": bool(карточка.единица_наблюдения),
        "Временная привязка подтверждена": bool(карточка.временная_привязка),
        "Признак доступен на момент решения": карточка.доступен_на_момент_решения is True,
        "Будущая информация не используется": карточка.использует_будущую_информацию is False,
        "Значение воспроизводимо для строк": карточка.воспроизводим_для_строк is True,
    }

    допущен = all(проверки.values())

    return {
        "Название признака": карточка.название,
        "Проверки": проверки,
        "Решение": "ДОПУЩЕН К ЭКСПЕРИМЕНТУ" if допущен else "НЕ ДОПУЩЕН К ЭКСПЕРИМЕНТУ",
    }


# Контрольный пример для текущего исторического датасета:
# внешний современный snapshot без row-level временной привязки.

контрольный_кандидат = КарточкаНовогоПризнака(
    название="Пример внешнего современного признака",
    источник="Внешний источник",
    описание="Современное состояние компании",
    правило_формирования="Получение текущего значения по INN",
    единица_наблюдения="Компания",
    временная_привязка=None,
    доступен_на_момент_решения=None,
    использует_будущую_информацию=None,
    воспроизводим_для_строк=False,
)

результат_допуска = проверить_допуск(контрольный_кандидат)

print("ПРОВЕРКА ДОПУСКА НОВОГО ПРИЗНАКА")
print("=" * 46)

for название, пройдено in результат_допуска["Проверки"].items():
    статус = "ДА" if пройдено else "НЕТ"
    print(f"{название}: {статус}")

print("\nРЕШЕНИЕ:")
print(результат_допуска["Решение"])

ПРОВЕРКА ДОПУСКА НОВОГО ПРИЗНАКА
Источник подтверждён: ДА
Описание подтверждено: ДА
Правило формирования подтверждено: ДА
Единица наблюдения подтверждена: ДА
Временная привязка подтверждена: НЕТ
Признак доступен на момент решения: НЕТ
Будущая информация не используется: НЕТ
Значение воспроизводимо для строк: НЕТ

РЕШЕНИЕ:
НЕ ДОПУЩЕН К ЭКСПЕРИМЕНТУ


### Результат проверки допуска

**ФАКТЫ**

Контрольный внешний признак не прошёл обязательный временной шлюз:

- источник известен;
- смысл признака известен;
- правило формирования известно;
- единица наблюдения известна;
- временная привязка к исторической строке отсутствует;
- нельзя подтвердить, что значение было доступно на момент принятия решения;
- нельзя подтвердить отсутствие информации из будущего;
- нельзя воспроизвести корректное историческое значение для каждой строки.

Итоговое решение:

**НЕ ДОПУЩЕН К ЭКСПЕРИМЕНТУ.**

**ИНТЕРПРЕТАЦИЯ**

Протокол корректно останавливает потенциально методологически неверный эксперимент до обучения модели.

Это защищает исследование от ситуации, когда современное состояние компании присоединяется к исторической строке по `INN` и ошибочно интерпретируется как информация, доступная в прошлом.

Такой признак нельзя честно сравнивать с текущим baseline на существующем историческом датасете.

**ОГРАНИЧЕНИЯ**

Этот результат не означает, что сам источник или признак бесполезен.

Не проверялись:

- Gini;
- ROC-AUC;
- PR-AUC;
- Recall;
- Precision;
- F1;
- ложноположительные ошибки;
- ложноотрицательные ошибки;
- влияние на 805 сложных дефолтов.

Причина остановки исключительно методологическая: отсутствует подтверждённая временная корректность.

**СЛЕДУЮЩИЙ ШАГ**

Зафиксировать разные типы решений внутри конвейера, чтобы не смешивать:

1. признак, который нельзя корректно проверить;
2. признак, который проверен и не дал пользы;
3. признак, который прошёл controlled experiment и может быть принят.

## 2. Возможные решения по новому признаку

Для каждого нового признака решение принимается поэтапно.

Важно разделять **методологический допуск** и **результат проверки качества**.

### Возможные статусы

**1. НЕ ДОПУЩЕН К ЭКСПЕРИМЕНТУ**

Используется, если отсутствует достаточная информация для честной проверки:

- неизвестно происхождение;
- неизвестно правило формирования;
- нет временной привязки;
- нельзя исключить использование будущей информации;
- невозможно воспроизвести значения для исследуемых строк.

Такой статус **не означает, что признак плохой**. Его predictive value остаётся неизвестным.

---

**2. ДОПУЩЕН К ЭКСПЕРИМЕНТУ**

Все методологические требования выполнены.

Только после этого разрешается выполнить controlled experiment:

> baseline → добавить один признак → тот же протокол проверки → сравнить результаты.

---

**3. ОТКЛОНЁН ПО РЕЗУЛЬТАТАМ ЭКСПЕРИМЕНТА**

Используется только после корректного controlled experiment, если новый признак не показывает достаточной дополнительной пользы или создаёт неприемлемые побочные эффекты.

---

**4. ПРИНЯТ**

Используется только после корректного controlled experiment, если evidence поддерживает включение нового признака.

Решение должно учитывать не только изменение общего Gini, но и:

- PR-AUC;
- Recall;
- Precision;
- F1;
- устойчивость по folds;
- изменение ложноположительных ошибок;
- изменение ложноотрицательных ошибок;
- поведение зоны дополнительной проверки;
- поведение сложных случаев;
- стоимость вычисления и воспроизводимость.

In [3]:
# 2.1. Формальные статусы признака внутри исследовательского конвейера

СТАТУСЫ_ПРИЗНАКА = {
    "НЕ_ДОПУЩЕН": {
        "русское_название": "НЕ ДОПУЩЕН К ЭКСПЕРИМЕНТУ",
        "смысл": (
            "Недостаточно подтверждений для методологически корректной проверки. "
            "Predictive value остаётся неизвестным."
        ),
        "обучение_разрешено": False,
    },
    "ДОПУЩЕН": {
        "русское_название": "ДОПУЩЕН К ЭКСПЕРИМЕНТУ",
        "смысл": (
            "Методологические требования выполнены. "
            "Разрешён один controlled experiment."
        ),
        "обучение_разрешено": True,
    },
    "ОТКЛОНЁН": {
        "русское_название": "ОТКЛОНЁН ПО РЕЗУЛЬТАТАМ ЭКСПЕРИМЕНТА",
        "смысл": (
            "Корректный эксперимент выполнен, "
            "но evidence не поддерживает включение признака."
        ),
        "обучение_разрешено": False,
    },
    "ПРИНЯТ": {
        "русское_название": "ПРИНЯТ",
        "смысл": (
            "Корректный эксперимент выполнен, "
            "и evidence поддерживает включение признака."
        ),
        "обучение_разрешено": False,
    },
}


print("СТАТУСЫ НОВОГО ПРИЗНАКА")
print("=" * 60)

for код, данные in СТАТУСЫ_ПРИЗНАКА.items():
    print(f"\n{данные['русское_название']}")
    print(f"  {данные['смысл']}")
    print(
        "  Обучение разрешено:"
        f" {'ДА' if данные['обучение_разрешено'] else 'НЕТ'}"
    )

СТАТУСЫ НОВОГО ПРИЗНАКА

НЕ ДОПУЩЕН К ЭКСПЕРИМЕНТУ
  Недостаточно подтверждений для методологически корректной проверки. Predictive value остаётся неизвестным.
  Обучение разрешено: НЕТ

ДОПУЩЕН К ЭКСПЕРИМЕНТУ
  Методологические требования выполнены. Разрешён один controlled experiment.
  Обучение разрешено: ДА

ОТКЛОНЁН ПО РЕЗУЛЬТАТАМ ЭКСПЕРИМЕНТА
  Корректный эксперимент выполнен, но evidence не поддерживает включение признака.
  Обучение разрешено: НЕТ

ПРИНЯТ
  Корректный эксперимент выполнен, и evidence поддерживает включение признака.
  Обучение разрешено: НЕТ


## 3. Контракт контролируемого эксперимента

После прохождения методологического допуска новый признак может быть проверен только в контролируемом эксперименте.

Цель такого эксперимента — ответить на один вопрос:

> **Даёт ли один новый признак дополнительную полезную информацию относительно принятого baseline при неизменных данных и протоколе оценки?**

### Единственное разрешённое изменение

В экспериментальную версию модели добавляется **ровно один новый признак**.

### Что запрещено менять одновременно

Во время проверки нового признака нельзя одновременно изменять:

- исходный датасет;
- `DefMark`;
- `INN`;
- working / final test split;
- folds;
- seed;
- остальные 47 разрешённых признаков;
- preprocessing существующих признаков;
- модель или её гиперпараметры;
- способ расчёта метрик;
- threshold;
- размер зоны дополнительной проверки.

Иначе невозможно определить, связано ли изменение результата именно с новым признаком.

### Уровень оценки

Выбор признака выполняется только на **working data** через тот же CV / OOF-протокол.

Final test остаётся закрытым и не используется:

- для отбора признака;
- для настройки параметров;
- для выбора threshold;
- для выбора зоны дополнительной проверки;
- для определения дальнейшего направления исследования.

### Обязательные результаты сравнения

Для baseline и версии с новым признаком должны быть рассчитаны на одинаковых OOF-наблюдениях:

- Gini / ROC-AUC;
- PR-AUC;
- Recall;
- Precision;
- F1;
- число ложноположительных ошибок;
- число ложноотрицательных ошибок;
- стабильность результата между folds;
- влияние на сложные случаи;
- влияние на ранее выделенную зону дополнительной проверки.

До получения результатов нельзя менять правила эксперимента в пользу нового признака.

In [4]:
# 3.1. Формальный контракт контролируемого эксперимента

from dataclasses import dataclass
from typing import Tuple


@dataclass(frozen=True)
class КонтрактЭксперимента:
    целевая_переменная: str
    идентификатор: str
    число_базовых_признаков: int

    новый_признак: str
    число_новых_признаков: int

    working_split_изменён: bool
    folds_изменены: bool
    seed_изменён: bool
    preprocessing_изменён: bool
    модель_изменена: bool
    гиперпараметры_изменены: bool
    threshold_изменён: bool
    final_test_использован: bool

    обязательные_метрики: Tuple[str, ...]


ОБЯЗАТЕЛЬНЫЕ_МЕТРИКИ = (
    "Gini",
    "ROC-AUC",
    "PR-AUC",
    "Recall",
    "Precision",
    "F1",
    "Ложноположительные ошибки",
    "Ложноотрицательные ошибки",
    "Стабильность по folds",
)


def проверить_контракт(
    контракт: КонтрактЭксперимента,
) -> dict:
    проверки = {
        "Целевая переменная неизменна":
            контракт.целевая_переменная == "DefMark",

        "Идентификатор неизменен":
            контракт.идентификатор == "INN",

        "Базовых признаков осталось 47":
            контракт.число_базовых_признаков == 47,

        "Добавляется ровно один новый признак":
            контракт.число_новых_признаков == 1,

        "Working split не изменён":
            not контракт.working_split_изменён,

        "Folds не изменены":
            not контракт.folds_изменены,

        "Seed не изменён":
            not контракт.seed_изменён,

        "Preprocessing baseline не изменён":
            not контракт.preprocessing_изменён,

        "Модель не изменена":
            not контракт.модель_изменена,

        "Гиперпараметры не изменены":
            not контракт.гиперпараметры_изменены,

        "Threshold не подбирался":
            not контракт.threshold_изменён,

        "Final test не использован":
            not контракт.final_test_использован,

        "Все обязательные метрики зафиксированы":
            set(ОБЯЗАТЕЛЬНЫЕ_МЕТРИКИ).issubset(
                контракт.обязательные_метрики
            ),
    }

    корректен = all(проверки.values())

    return {
        "Проверки": проверки,
        "Решение": (
            "КОНТРАКТ ЗАФИКСИРОВАН"
            if корректен
            else "КОНТРАКТ НАРУШЕН"
        ),
    }


контрольный_контракт = КонтрактЭксперимента(
    целевая_переменная="DefMark",
    идентификатор="INN",
    число_базовых_признаков=47,

    новый_признак="Пример допустимого нового признака",
    число_новых_признаков=1,

    working_split_изменён=False,
    folds_изменены=False,
    seed_изменён=False,
    preprocessing_изменён=False,
    модель_изменена=False,
    гиперпараметры_изменены=False,
    threshold_изменён=False,
    final_test_использован=False,

    обязательные_метрики=ОБЯЗАТЕЛЬНЫЕ_МЕТРИКИ,
)

результат_контракта = проверить_контракт(
    контрольный_контракт
)

print("ПРОВЕРКА КОНТРАКТА ЭКСПЕРИМЕНТА")
print("=" * 52)

for название, пройдено in результат_контракта["Проверки"].items():
    print(f"{название}: {'ДА' if пройдено else 'НЕТ'}")

print("\nРЕШЕНИЕ:")
print(результат_контракта["Решение"])

ПРОВЕРКА КОНТРАКТА ЭКСПЕРИМЕНТА
Целевая переменная неизменна: ДА
Идентификатор неизменен: ДА
Базовых признаков осталось 47: ДА
Добавляется ровно один новый признак: ДА
Working split не изменён: ДА
Folds не изменены: ДА
Seed не изменён: ДА
Preprocessing baseline не изменён: ДА
Модель не изменена: ДА
Гиперпараметры не изменены: ДА
Threshold не подбирался: ДА
Final test не использован: ДА
Все обязательные метрики зафиксированы: ДА

РЕШЕНИЕ:
КОНТРАКТ ЗАФИКСИРОВАН


### Результат фиксации контракта

**ФАКТЫ**

Контрольный контракт прошёл все обязательные проверки:

- целевая переменная и идентификатор не меняются;
- базовая версия модели сохраняет 47 разрешённых признаков;
- экспериментальная версия получает ровно один дополнительный признак;
- рабочая выборка, разбиение на части и случайное начальное состояние остаются прежними;
- модель, её гиперпараметры и предварительная обработка базовой версии не меняются;
- финальная тестовая выборка не используется;
- обязательный набор метрик зафиксирован до начала эксперимента.

Итог:

**КОНТРАКТ ЗАФИКСИРОВАН.**

**ИНТЕРПРЕТАЦИЯ**

Такой контракт позволяет связать наблюдаемое изменение результата именно с добавлением нового признака, а не со сменой модели, разбиения данных или параметров обучения.

Для анализа ошибок фиксируется одинаковое бизнес-правило принятия решений. Например, если используется режим, при котором 15% клиентов относятся к зоне повышенного риска, эта доля должна оставаться одинаковой для базовой и экспериментальной версии модели.

Числовой порог вероятности при этом может отличаться между моделями, поскольку шкала итоговой оценки риска может измениться. Такой порог определяется механически из заранее выбранной доли решений и не подбирается по результатам эксперимента.

**ОГРАНИЧЕНИЯ**

Сам по себе контракт ещё ничего не говорит о полезности нового признака.

Он определяет только условия, при которых будущий результат можно будет корректно интерпретировать как контролируемое сравнение.

**СЛЕДУЮЩИЙ ШАГ**

Зафиксировать единый набор показателей для сравнения базовой модели и модели с новым признаком, а также отделить наблюдаемые изменения метрик от окончательного решения о принятии признака.

In [5]:
# 4.1. Универсальный пакет сравнения baseline и нового признака

from dataclasses import dataclass, asdict
from typing import Optional, Tuple


@dataclass(frozen=True)
class РезультатМодели:
    gini: float
    roc_auc: float
    pr_auc: float
    recall: float
    precision: float
    f1: float
    ложноположительные: int
    ложноотрицательные: int
    fold_gini: Tuple[float, ...]


@dataclass(frozen=True)
class ДиагностикаСложныхСлучаев:
    всего_сложных_дефолтов: int
    захвачено_baseline: int
    захвачено_с_новым_признаком: int


@dataclass(frozen=True)
class ДиагностикаЗоныПроверки:
    доля_выборки: float
    ошибок_baseline: int
    ошибок_с_новым_признаком: int
    ложноотрицательных_baseline: int
    ложноотрицательных_с_новым_признаком: int
    ложноположительных_baseline: int
    ложноположительных_с_новым_признаком: int


def собрать_сравнение(
    baseline: РезультатМодели,
    новый: РезультатМодели,
    сложные: Optional[ДиагностикаСложныхСлучаев] = None,
    зона: Optional[ДиагностикаЗоныПроверки] = None,
) -> dict:

    if len(baseline.fold_gini) != len(новый.fold_gini):
        raise ValueError(
            "Baseline и новый признак должны быть проверены "
            "на одинаковом числе folds."
        )

    fold_delta = tuple(
        new - base
        for base, new in zip(
            baseline.fold_gini,
            новый.fold_gini,
        )
    )

    сравнение = {
        "Общее качество": {
            "Δ Gini": новый.gini - baseline.gini,
            "Δ ROC-AUC": новый.roc_auc - baseline.roc_auc,
            "Δ PR-AUC": новый.pr_auc - baseline.pr_auc,
            "Δ Recall": новый.recall - baseline.recall,
            "Δ Precision": новый.precision - baseline.precision,
            "Δ F1": новый.f1 - baseline.f1,
        },

        "Ошибки": {
            "Δ ложноположительных":
                новый.ложноположительные
                - baseline.ложноположительные,

            "Δ ложноотрицательных":
                новый.ложноотрицательные
                - baseline.ложноотрицательные,
        },

        "Стабильность": {
            "Δ Gini по folds": fold_delta,
            "Улучшений по folds":
                sum(delta > 0 for delta in fold_delta),
            "Ухудшений по folds":
                sum(delta < 0 for delta in fold_delta),
        },

        "Сложные случаи": (
            asdict(сложные)
            if сложные is not None
            else None
        ),

        "Зона дополнительной проверки": (
            asdict(зона)
            if зона is not None
            else None
        ),
    }

    return сравнение


print("ПАКЕТ СРАВНЕНИЯ НОВОГО ПРИЗНАКА")
print("=" * 52)
print("Структура подготовлена.")
print()
print("В пакет входят:")
print("- общее качество;")
print("- ложноположительные и ложноотрицательные ошибки;")
print("- стабильность по folds;")
print("- сложные случаи;")
print("- зона дополнительной проверки.")
print()
print(
    "Автоматический порог ACCEPT / REJECT "
    "не задан до появления утверждённого критерия."
)

ПАКЕТ СРАВНЕНИЯ НОВОГО ПРИЗНАКА
Структура подготовлена.

В пакет входят:
- общее качество;
- ложноположительные и ложноотрицательные ошибки;
- стабильность по folds;
- сложные случаи;
- зона дополнительной проверки.

Автоматический порог ACCEPT / REJECT не задан до появления утверждённого критерия.


### Результат подготовки пакета сравнения

**ФАКТЫ**

Подготовлена единая структура сравнения базовой модели и модели с одним новым признаком.

В неё входят:

- общее качество модели;
- ложноположительные и ложноотрицательные ошибки;
- устойчивость результата между частями перекрёстной проверки;
- поведение сложных случаев;
- поведение зоны дополнительной проверки.

Автоматический порог для принятия или отклонения признака заранее не задан.

**ИНТЕРПРЕТАЦИЯ**

Новый признак нельзя принимать только потому, что одна метрика немного выросла.

Решение должно учитывать совокупность результатов:

- изменение общего качества;
- устойчивость улучшения;
- изменение характера ошибок;
- влияние на сложные случаи;
- влияние на дополнительную проверку.

При этом числовые критерии нельзя придумывать после просмотра результата.

**ОГРАНИЧЕНИЯ**

На этом шаге реальный новый признак ещё не проверяется.

Поэтому структура сравнения не содержит фактических результатов эксперимента и не позволяет сделать вывод о принятии или отклонении конкретного признака.

**СЛЕДУЮЩИЙ ШАГ**

Зафиксировать правило принятия решения так, чтобы отсутствие заранее утверждённого критерия приводило не к произвольному выбору, а к статусу «недостаточно оснований для решения».

## 5. Правило принятия решения по новому признаку

После завершения контролируемого эксперимента необходимо отдельно принять решение о судьбе нового признака.

Важно не смешивать два разных этапа:

1. **расчёт результатов эксперимента**;
2. **интерпретацию этих результатов и принятие решения**.

### Возможные решения

**ПРИНЯТ**

Признак может быть принят только тогда, когда:

- он прошёл методологический допуск;
- эксперимент выполнен по зафиксированному контракту;
- критерий принятия был определён до просмотра результатов;
- результаты удовлетворяют этому критерию;
- отсутствуют существенные противоречия между общим качеством, устойчивостью и структурой ошибок.

---

**ОТКЛОНЁН**

Признак может быть отклонён, если корректный эксперимент показывает отсутствие дополнительной пользы либо устойчивое ухудшение по заранее зафиксированным критериям.

---

**НЕДОСТАТОЧНО ОСНОВАНИЙ ДЛЯ РЕШЕНИЯ**

Этот статус используется, если:

- изменения слишком малы или нестабильны;
- разные показатели дают противоречивую картину;
- отсутствует заранее утверждённый числовой критерий;
- имеющихся результатов недостаточно для однозначного решения.

Такой статус предпочтительнее произвольного выбора после просмотра результата.

### Основной принцип

Если критерий принятия не был определён заранее, код не должен автоматически объявлять новый признак «хорошим» или «плохим».

В этом случае корректный вывод:

**НЕДОСТАТОЧНО ОСНОВАНИЙ ДЛЯ РЕШЕНИЯ.**

In [7]:
# 5.1. Формальное правило принятия решения по новому признаку

from dataclasses import dataclass


@dataclass(frozen=True)
class ОснованияДляРешения:
    методологический_допуск_пройден: bool
    эксперимент_выполнен: bool
    контракт_соблюдён: bool
    критерий_принятия_задан_заранее: bool

    критерий_принятия_выполнен: bool | None
    критерий_отклонения_выполнен: bool | None

    есть_существенные_противоречия: bool


def принять_решение(
    основания: ОснованияДляРешения,
) -> str:

    if not основания.методологический_допуск_пройден:
        return "НЕ ДОПУЩЕН К ЭКСПЕРИМЕНТУ"

    if not основания.эксперимент_выполнен:
        return "ЭКСПЕРИМЕНТ ЕЩЁ НЕ ВЫПОЛНЕН"

    if not основания.контракт_соблюдён:
        return "РЕЗУЛЬТАТ ЭКСПЕРИМЕНТА НЕКОРРЕКТЕН"

    if not основания.критерий_принятия_задан_заранее:
        return "НЕДОСТАТОЧНО ОСНОВАНИЙ ДЛЯ РЕШЕНИЯ"

    if основания.есть_существенные_противоречия:
        return "НЕДОСТАТОЧНО ОСНОВАНИЙ ДЛЯ РЕШЕНИЯ"

    if основания.критерий_принятия_выполнен is True:
        return "ПРИНЯТ"

    if основания.критерий_отклонения_выполнен is True:
        return "ОТКЛОНЁН"

    return "НЕДОСТАТОЧНО ОСНОВАНИЙ ДЛЯ РЕШЕНИЯ"


контрольный_пример = ОснованияДляРешения(
    методологический_допуск_пройден=True,
    эксперимент_выполнен=True,
    контракт_соблюдён=True,

    # Для текущего протокола специальный числовой
    # критерий принятия ещё не утверждён.
    критерий_принятия_задан_заранее=False,

    критерий_принятия_выполнен=None,
    критерий_отклонения_выполнен=None,

    есть_существенные_противоречия=False,
)

решение = принять_решение(контрольный_пример)

print("ПРОВЕРКА ПРАВИЛА ПРИНЯТИЯ РЕШЕНИЯ")
print("=" * 50)
print()
print("Критерий принятия задан заранее: НЕТ")
print()
print("РЕШЕНИЕ:")
print(решение)

ПРОВЕРКА ПРАВИЛА ПРИНЯТИЯ РЕШЕНИЯ

Критерий принятия задан заранее: НЕТ

РЕШЕНИЕ:
НЕДОСТАТОЧНО ОСНОВАНИЙ ДЛЯ РЕШЕНИЯ


### Результат проверки правила принятия решения

**ФАКТЫ**

Контрольный пример прошёл методологический допуск, эксперимент считается корректно выполненным, а контракт — соблюдённым.

При этом заранее утверждённый критерий принятия признака отсутствует.

Итоговое решение:

**НЕДОСТАТОЧНО ОСНОВАНИЙ ДЛЯ РЕШЕНИЯ.**

**ИНТЕРПРЕТАЦИЯ**

Протокол не позволяет автоматически принять новый признак только после просмотра полученных метрик.

Если критерий принятия не был определён до эксперимента, результат можно описать и проанализировать, но нельзя задним числом подобрать удобную границу и объявить признак успешным.

Это отделяет фактический результат эксперимента от управленческого решения о включении признака в рабочую модель.

**ОГРАНИЧЕНИЯ**

На этом шаге не проверяется конкретный новый признак и не задаётся универсальная числовая граница полезности.

Такая граница может появиться только как отдельное заранее принятое исследовательское или бизнес-решение.

**СЛЕДУЮЩИЙ ШАГ**

Проверить весь протокол целиком на нескольких контрольных сценариях и убедиться, что каждый тип ситуации приводит к ожидаемому решению.

## 6. Сквозная проверка протокола

Теперь необходимо убедиться, что отдельные правила образуют единый последовательный процесс.

Для этого используются контрольные сценарии без обучения модели и без реального нового признака.

Проверяются четыре принципиально разные ситуации:

1. признак не прошёл временной допуск;
2. признак прошёл допуск, но эксперимент ещё не выполнен;
3. эксперимент выполнен корректно, но заранее утверждённого критерия решения нет;
4. эксперимент выполнен корректно и заранее заданный критерий выполнен.

Цель этой проверки — убедиться, что протокол не пропускает методологически некорректный признак и не принимает решение раньше, чем для этого появились необходимые основания.

На этом шаге:

- модель не обучается;
- новые признаки не создаются;
- финальная тестовая выборка не используется;
- реальные показатели качества не рассчитываются.

In [8]:
# 6.1. Сквозная проверка логики протокола

контрольные_сценарии = {
    "Нет временного допуска": ОснованияДляРешения(
        методологический_допуск_пройден=False,
        эксперимент_выполнен=False,
        контракт_соблюдён=False,
        критерий_принятия_задан_заранее=False,
        критерий_принятия_выполнен=None,
        критерий_отклонения_выполнен=None,
        есть_существенные_противоречия=False,
    ),

    "Допущен, но эксперимент не выполнен": ОснованияДляРешения(
        методологический_допуск_пройден=True,
        эксперимент_выполнен=False,
        контракт_соблюдён=True,
        критерий_принятия_задан_заранее=False,
        критерий_принятия_выполнен=None,
        критерий_отклонения_выполнен=None,
        есть_существенные_противоречия=False,
    ),

    "Эксперимент выполнен, критерия нет": ОснованияДляРешения(
        методологический_допуск_пройден=True,
        эксперимент_выполнен=True,
        контракт_соблюдён=True,
        критерий_принятия_задан_заранее=False,
        критерий_принятия_выполнен=None,
        критерий_отклонения_выполнен=None,
        есть_существенные_противоречия=False,
    ),

    "Критерий задан заранее и выполнен": ОснованияДляРешения(
        методологический_допуск_пройден=True,
        эксперимент_выполнен=True,
        контракт_соблюдён=True,
        критерий_принятия_задан_заранее=True,
        критерий_принятия_выполнен=True,
        критерий_отклонения_выполнен=False,
        есть_существенные_противоречия=False,
    ),
}


ожидаемые_решения = {
    "Нет временного допуска":
        "НЕ ДОПУЩЕН К ЭКСПЕРИМЕНТУ",

    "Допущен, но эксперимент не выполнен":
        "ЭКСПЕРИМЕНТ ЕЩЁ НЕ ВЫПОЛНЕН",

    "Эксперимент выполнен, критерия нет":
        "НЕДОСТАТОЧНО ОСНОВАНИЙ ДЛЯ РЕШЕНИЯ",

    "Критерий задан заранее и выполнен":
        "ПРИНЯТ",
}


print("СКВОЗНАЯ ПРОВЕРКА ПРОТОКОЛА")
print("=" * 56)

все_проверки_пройдены = True

for название, основания in контрольные_сценарии.items():
    фактическое = принять_решение(основания)
    ожидаемое = ожидаемые_решения[название]

    совпадает = фактическое == ожидаемое
    все_проверки_пройдены &= совпадает

    print(f"\nСценарий: {название}")
    print(f"Ожидаемое решение:  {ожидаемое}")
    print(f"Фактическое решение: {фактическое}")
    print(f"Проверка: {'ПРОЙДЕНА' if совпадает else 'НЕ ПРОЙДЕНА'}")


print("\n" + "=" * 56)

if все_проверки_пройдены:
    print("ИТОГ: ВСЕ КОНТРОЛЬНЫЕ СЦЕНАРИИ ПРОЙДЕНЫ")
else:
    print("ИТОГ: ОБНАРУЖЕНО НАРУШЕНИЕ ЛОГИКИ ПРОТОКОЛА")

СКВОЗНАЯ ПРОВЕРКА ПРОТОКОЛА

Сценарий: Нет временного допуска
Ожидаемое решение:  НЕ ДОПУЩЕН К ЭКСПЕРИМЕНТУ
Фактическое решение: НЕ ДОПУЩЕН К ЭКСПЕРИМЕНТУ
Проверка: ПРОЙДЕНА

Сценарий: Допущен, но эксперимент не выполнен
Ожидаемое решение:  ЭКСПЕРИМЕНТ ЕЩЁ НЕ ВЫПОЛНЕН
Фактическое решение: ЭКСПЕРИМЕНТ ЕЩЁ НЕ ВЫПОЛНЕН
Проверка: ПРОЙДЕНА

Сценарий: Эксперимент выполнен, критерия нет
Ожидаемое решение:  НЕДОСТАТОЧНО ОСНОВАНИЙ ДЛЯ РЕШЕНИЯ
Фактическое решение: НЕДОСТАТОЧНО ОСНОВАНИЙ ДЛЯ РЕШЕНИЯ
Проверка: ПРОЙДЕНА

Сценарий: Критерий задан заранее и выполнен
Ожидаемое решение:  ПРИНЯТ
Фактическое решение: ПРИНЯТ
Проверка: ПРОЙДЕНА

ИТОГ: ВСЕ КОНТРОЛЬНЫЕ СЦЕНАРИИ ПРОЙДЕНЫ


### Результат сквозной проверки протокола

**ФАКТЫ**

Все четыре заранее заданных контрольных сценария отработали ожидаемо:

1. признак без подтверждённой временной корректности не допускается к эксперименту;
2. допущенный признак без выполненного эксперимента не получает итогового решения;
3. корректно выполненный эксперимент без заранее заданного критерия не приводит к произвольному принятию или отклонению;
4. при заранее заданном и выполненном критерии признак может получить статус «ПРИНЯТ».

Все фактические решения совпали с ожидаемыми.

Итог:

**ВСЕ КОНТРОЛЬНЫЕ СЦЕНАРИИ ПРОЙДЕНЫ.**

**ИНТЕРПРЕТАЦИЯ**

Отдельные правила объединены в единый последовательный процесс:

**источник признака → проверка происхождения → проверка временной корректности → допуск → одно контролируемое изменение → одинаковый протокол оценки → анализ качества и ошибок → решение.**

Протокол разделяет три принципиально разные ситуации:

- признак нельзя корректно проверить;
- признак можно проверить, но доказательств для решения пока недостаточно;
- корректный эксперимент даёт достаточные основания для принятия или отклонения.

Это снижает риск методологических ошибок и принятия решений после просмотра удобных метрик.

**ОГРАНИЧЕНИЯ**

Сквозная проверка проверяет логику исследовательского процесса, а не predictive value конкретного нового признака.

На этом этапе:

- новый признак не создавался;
- модель не обучалась;
- качество модели не пересчитывалось;
- финальная тестовая выборка не использовалась;
- временная устойчивость модели не проверялась.

Протокол также не определяет универсальный числовой критерий полезности нового признака. Такой критерий должен быть зафиксирован отдельно до конкретного эксперимента.

**СЛЕДУЮЩИЙ ШАГ**

Сохранить воспроизводимый артефакт Stage 19 и зафиксировать протокол как готовую основу для будущих экспериментов с новыми признаками.

## 7. Итог этапа и сохранение протокола

Этап 19 не является экспериментом с новой моделью или новым признаком.

Его результат — воспроизводимый исследовательский протокол, который можно применять к будущим признакам без изменения основных правил оценки.

В итоговом артефакте фиксируются:

- исследовательский вопрос;
- обязательный временной допуск;
- контракт одного контролируемого изменения;
- обязательный набор показателей;
- правила анализа ошибок;
- возможные статусы признака;
- правило принятия решения;
- результаты контрольной проверки логики;
- подтверждение того, что модель не обучалась и финальная тестовая выборка не использовалась.

In [9]:
# 7.1. Сохранение воспроизводимого артефакта Stage 19

import json
from pathlib import Path


путь_артефакта = Path(
    "../reports/generated/stage19_new_feature_protocol_V1.json"
)

путь_артефакта.parent.mkdir(
    parents=True,
    exist_ok=True,
)


результаты_сценариев = []

for название, основания in контрольные_сценарии.items():
    фактическое = принять_решение(основания)
    ожидаемое = ожидаемые_решения[название]

    результаты_сценариев.append(
        {
            "сценарий": название,
            "ожидаемое_решение": ожидаемое,
            "фактическое_решение": фактическое,
            "проверка_пройдена": фактическое == ожидаемое,
        }
    )


артефакт = {
    "stage": "Stage 19",
    "version": "V1",
    "status": "NEW_FEATURE_PROTOCOL_READY",

    "research_question": (
        "Как должен быть устроен единый воспроизводимый процесс "
        "проверки нового признака, чтобы сравнение с текущей моделью "
        "оставалось честным, не возникала временная утечка, "
        "а решение учитывало не только общее качество, "
        "но и характер ошибок?"
    ),

    "experiment_flags": {
        "new_feature_created": False,
        "model_training": False,
        "hyperparameter_tuning": False,
        "threshold_optimization": False,
        "final_test_used": False,
    },

    "baseline_contract": {
        "target": "DefMark",
        "identifier": "INN",
        "allowed_features": 47,
        "Q_B1_norm_used_as_predictor": False,
        "Q_B2_norm_used_as_predictor": False,
        "working_split_changed": False,
        "folds_changed": False,
        "seed_changed": False,
    },

    "admission_gate": [
        "Источник данных подтверждён",
        "Описание признака подтверждено",
        "Правило формирования подтверждено",
        "Единица наблюдения подтверждена",
        "Временная привязка подтверждена",
        "Признак доступен на момент решения",
        "Будущая информация не используется",
        "Значение воспроизводимо для исследуемых строк",
    ],

    "controlled_change": {
        "new_features_per_experiment": 1,
        "model_changed": False,
        "baseline_preprocessing_changed": False,
        "hyperparameters_changed": False,
    },

    "required_metrics": list(
        ОБЯЗАТЕЛЬНЫЕ_МЕТРИКИ
    ),

    "required_diagnostics": [
        "Ложноположительные ошибки",
        "Ложноотрицательные ошибки",
        "Стабильность по folds",
        "Сложные случаи",
        "Зона дополнительной проверки",
    ],

    "feature_statuses": [
        "НЕ ДОПУЩЕН К ЭКСПЕРИМЕНТУ",
        "ДОПУЩЕН К ЭКСПЕРИМЕНТУ",
        "ОТКЛОНЁН ПО РЕЗУЛЬТАТАМ ЭКСПЕРИМЕНТА",
        "ПРИНЯТ",
        "НЕДОСТАТОЧНО ОСНОВАНИЙ ДЛЯ РЕШЕНИЯ",
    ],

    "decision_rule": {
        "criterion_must_be_defined_before_results": True,
        "automatic_accept_without_predefined_criterion": False,
        "insufficient_evidence_status_required": True,
    },

    "control_scenarios": результаты_сценариев,

    "control_scenarios_all_passed": all(
        строка["проверка_пройдена"]
        for строка in результаты_сценариев
    ),

    "limitations": [
        "Конкретный новый признак не проверялся",
        "Predictive value нового признака не оценивался",
        "Модель не переобучалась",
        "Final test не использовался",
        "Temporal stability не проверялась",
        "Универсальный числовой критерий принятия не установлен",
    ],

    "next_use": (
        "Применять протокол к будущему признаку только после "
        "подтверждения его происхождения и временной корректности."
    ),
}


assert артефакт["control_scenarios_all_passed"] is True
assert артефакт["experiment_flags"]["model_training"] is False
assert артефакт["experiment_flags"]["final_test_used"] is False
assert артефакт["controlled_change"]["new_features_per_experiment"] == 1


with путь_артефакта.open(
    "w",
    encoding="utf-8",
) as файл:
    json.dump(
        артефакт,
        файл,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )


# Повторно читаем сохранённый файл:
# это проверяет, что получился корректный JSON.

with путь_артефакта.open(
    "r",
    encoding="utf-8",
) as файл:
    проверка = json.load(файл)


assert проверка["status"] == "NEW_FEATURE_PROTOCOL_READY"
assert проверка["control_scenarios_all_passed"] is True
assert проверка["experiment_flags"]["final_test_used"] is False


print("АРТЕФАКТ ЭТАПА 19")
print("=" * 52)
print(f"Файл: {путь_артефакта}")
print(f"Статус: {проверка['status']}")
print(
    "Контрольные сценарии пройдены:",
    "ДА" if проверка["control_scenarios_all_passed"] else "НЕТ",
)
print(
    "Модель обучалась:",
    "ДА" if проверка["experiment_flags"]["model_training"] else "НЕТ",
)
print(
    "Финальная тестовая выборка использовалась:",
    "ДА" if проверка["experiment_flags"]["final_test_used"] else "НЕТ",
)
print()
print("ПРОВЕРКА АРТЕФАКТА: OK")

АРТЕФАКТ ЭТАПА 19
Файл: ..\reports\generated\stage19_new_feature_protocol_V1.json
Статус: NEW_FEATURE_PROTOCOL_READY
Контрольные сценарии пройдены: ДА
Модель обучалась: НЕТ
Финальная тестовая выборка использовалась: НЕТ

ПРОВЕРКА АРТЕФАКТА: OK


## 8. Итог этапа 19

### ФАКТЫ

На этапе 19 сформирован и проверен единый воспроизводимый протокол работы с новым признаком.

Протокол требует следующую последовательность:

**источник → происхождение → временная корректность → методологический допуск → одно контролируемое изменение → одинаковая проверка качества → анализ ошибок → анализ сложных случаев → решение.**

Подтверждено:

- признак без подтверждённой временной корректности останавливается до обучения модели;
- в одном эксперименте разрешено добавлять ровно один новый признак;
- целевая переменная, идентификатор, рабочая выборка, разбиение, модель и параметры обучения должны оставаться неизменными;
- финальная тестовая выборка не используется для выбора признака;
- результат оценивается не только по общему качеству, но и по ложноположительным и ложноотрицательным ошибкам, устойчивости, сложным случаям и зоне дополнительной проверки;
- без заранее заданного критерия протокол не позволяет автоматически принять или отклонить признак;
- все четыре контрольных сценария работы протокола прошли успешно.

На этом этапе:

- новый реальный признак не добавлялся;
- модель не переобучалась;
- гиперпараметры не подбирались;
- порог решения не оптимизировался;
- финальная тестовая выборка не использовалась.

Статус этапа:

**`NEW_FEATURE_PROTOCOL_READY`**

### ИНТЕРПРЕТАЦИЯ

Полученный результат превращает проверку нового признака из разового эксперимента в повторяемый исследовательский процесс.

Теперь будущий признак нельзя просто добавить в таблицу и проверить, выросла ли одна метрика. Сначала необходимо доказать, что признак методологически допустим, после чего выполнить одно контролируемое сравнение на неизменном протоколе.

Такой подход позволяет отдельно различать:

1. признак, который нельзя корректно проверить;
2. признак, который можно проверить, но имеющихся результатов недостаточно для решения;
3. признак, который корректно проверен и получил достаточные основания для принятия или отклонения.

Это соответствует практической задаче проекта: не бесконечно искать новые модели, а создать управляемый процесс проверки новых источников информации и новых признаков.

### ОГРАНИЧЕНИЯ

Этап 19 не доказывает полезность какого-либо конкретного нового признака.

Также он не устанавливает:

- универсальную минимальную прибавку Gini;
- универсальный допустимый уровень Recall или Precision;
- оптимальную стоимость ложноположительной и ложноотрицательной ошибки;
- оптимальный размер зоны дополнительной проверки;
- временную устойчивость будущей модели.

Такие критерии должны определяться отдельно и до просмотра результатов соответствующего эксперимента.

Для текущего исторического датасета сохраняется ранее установленное ограничение: современные внешние данные нельзя корректно присоединять к старым строкам без надёжной построчной временной привязки.

### СЛЕДУЮЩИЙ ШАГ

Использовать протокол как стандартную точку входа для будущего нового признака.

Практическое применение становится возможным, когда появляется новый набор данных или источник, для которого подтверждены:

- момент наблюдения;
- доступность информации на этот момент;
- происхождение признака;
- возможность воспроизвести его значение без использования будущей информации.

До появления такого объекта данных дополнительное обучение на текущем историческом датасете не требуется.